In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import *
from sklearn import metrics
import statsmodels.api as sm
from yellowbrick.classifier import DiscriminationThreshold
import plotly.graph_objects as go
import plotly.express as px
import plotly.offline as py
from plotly.subplots import make_subplots

In [ ]:
df =pd.read_csv('../data/Mod_tel_churn.csv')
df.head()

In [ ]:
def get_selected_feature_names(df, k=10):
    X = df.drop(columns="Churn")
    y = df["Churn"]
    # For Pandas 3.0 compatibility, include 'string' and 'object'
    categorical_columns = X.select_dtypes(include=['object', 'string', 'category']).columns
    if not categorical_columns.empty:
        X_encoded = X.copy()
        for col in categorical_columns:
            X_encoded[col] = LabelEncoder().fit_transform(X[col].astype(str))
    else:
        X_encoded = X
    
    selected_features = SelectKBest(chi2, k=k)
    selected_features.fit(X_encoded, y)
    
    selected_mask = selected_features.get_support()
    selected_feature_names = X.columns[selected_mask]
    
    return list(selected_feature_names)


In [ ]:
selected_feature_names = get_selected_feature_names(df,k=10)
print(selected_feature_names)

In [ ]:
df = df[selected_feature_names + ['Churn']]
df_og = df.copy()
df.head()

In [ ]:
df.sample()

In [ ]:
df.to_csv('../data/data-predect.csv')

In [ ]:
print(df.select_dtypes(include=['object']).columns)

In [ ]:
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})
df.head()

In [ ]:
import pickle
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np

def preprocess_data(df):
    # Create a copy to avoid SettingWithCopyWarning
    df_work = df.copy()
    
    # 1. Force convert TotalCharges to numeric if it exists
    if 'TotalCharges' in df_work.columns:
        df_work['TotalCharges'] = pd.to_numeric(df_work['TotalCharges'], errors='coerce')
        df_work['TotalCharges'] = df_work['TotalCharges'].fillna(0)
    
    # 2. Separate Churn
    y = df_work['Churn']
    if y.dtype == 'object' or y.dtype == 'string':
        y = y.map({'No': 0, 'Yes': 1}).fillna(0)
    
    X = df_work.drop(columns=['Churn'])
    
    # 3. Identify categorical columns (anything not numeric)
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
    
    # 4. Scale all numeric columns
    scaler = StandardScaler()
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    if not numeric_cols.empty:
        X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    
    return X, y, label_encoders, scaler

# --- EXECUTION ---
X_proc, y_proc, label_encoders, scaler = preprocess_data(df)
df_processed = X_proc.copy()
df_processed['Churn'] = y_proc

with open('../models/label_encoders_and_scaler.pkl', 'wb') as f:
    pickle.dump((label_encoders, scaler), f)

print("Preprocessing complete. Columns remaining as objects:", df_processed.select_dtypes(exclude=[np.number]).columns.tolist())
df_processed.head()


In [ ]:
X = df_processed.drop(columns = "Churn")
y = df_processed["Churn"]
cols = X.columns.tolist()
X.head()

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

## **Appliquer à l'algorithme de machine learning :**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
def telecom_churn_prediction(algorithm, training_x, testing_x, training_y, testing_y, cf, threshold_plot):
    #model
    algorithm.fit(training_x, training_y)
    predictions = algorithm.predict(testing_x)
    probabilities = algorithm.predict_proba(testing_x)
        
    print('Algorithm:', type(algorithm).__name__)
    print("\nClassification report:\n", classification_report(testing_y, predictions))
    print("Accuracy Score:", accuracy_score(testing_y, predictions))
    
    #confusion matrix
    conf_matrix = confusion_matrix(testing_y, predictions)
    #roc_auc_score
    model_roc_auc = roc_auc_score(testing_y, predictions) 
    print("Area under curve:", model_roc_auc,"\n")
    
    fpr, tpr, thresholds = roc_curve(testing_y, probabilities[:,1])
     
    #plot confusion matrix
    trace1 = go.Heatmap(z = conf_matrix,
                        x = ["Not churn", "Churn"],
                        y = ["Not churn", "Churn"],
                        showscale = False, colorscale = "Picnic",
                        name = "Confusion matrix")
    #plot roc curve
    trace2 = go.Scatter(x = fpr, y = tpr,
                        name = "Roc: " + str(model_roc_auc),
                        line = dict(color = ('rgb(22, 96, 167)'), width = 2))
    trace3 = go.Scatter(x = [0,1], y = [0,1],
                        line = dict(color = ('rgb(205, 12, 24)'), width = 2,
                        dash = 'dot'))
    
    if cf in ['coefficients', 'features']:
        if cf == 'coefficients':
            coefficients = pd.DataFrame(algorithm.coef_.ravel())
        elif cf == 'features':
            coefficients = pd.DataFrame(algorithm.feature_importances_)
        
        column_df = pd.DataFrame(training_x.columns.tolist())
        coef_sumry = (pd.merge(coefficients, column_df, left_index=True, 
                               right_index=True, how="left"))
        coef_sumry.columns = ["coefficients", "features"]
        coef_sumry = coef_sumry.sort_values(by = "coefficients", ascending=False)
         #plot coeffs
        trace4 = go.Bar(x = coef_sumry["features"], y = coef_sumry["coefficients"], 
                        name = "coefficients",
                        marker = dict(color = coef_sumry["coefficients"],
                                      colorscale = "Picnic",
                                      line = dict(width = .6, color = "black")
                                     )
                       )
        #subplots
        fig = make_subplots(rows=2, cols=2, specs=[[{}, {}], [{'colspan': 2}, None]],
                                subplot_titles=('Confusion matrix',
                                                'Receiver operating characteristic',
                                                'Feature importances')
                           )  
        fig.append_trace(trace1,1,1)
        fig.append_trace(trace2,1,2)
        fig.append_trace(trace3,1,2)
        fig.append_trace(trace4,2,1)
        fig['layout'].update(showlegend=False, title="Model performance",
                             autosize=False, height = 900, width = 800,
                             plot_bgcolor = 'rgba(240,240,240, 0.95)',
                             paper_bgcolor = 'rgba(240,240,240, 0.95)',
                             margin = dict(b = 195))
        fig["layout"]["xaxis2"].update(dict(title = "false positive rate"))
        fig["layout"]["yaxis2"].update(dict(title = "true positive rate"))
        fig["layout"]["xaxis3"].update(dict(showgrid = True, tickfont = dict(size = 10), tickangle = 90))
        
    elif cf == 'None':
        #subplots
        fig = make_subplots(rows=1, cols=2,
                            subplot_titles=('Confusion matrix',
                                            'Receiver operating characteristic')
                           )
        fig.append_trace(trace1,1,1)
        fig.append_trace(trace2,1,2)
        fig.append_trace(trace3,1,2)
        fig['layout'].update(showlegend=False, title="Model performance",
                         autosize=False, height = 500, width = 800,
                         plot_bgcolor = 'rgba(240,240,240,0.95)',
                         paper_bgcolor = 'rgba(240,240,240,0.95)',
                         margin = dict(b = 195))
        fig["layout"]["xaxis2"].update(dict(title = "false positive rate"))
        fig["layout"]["yaxis2"].update(dict(title = "true positive rate"))  
        
    py.iplot(fig)
    
    if threshold_plot == True: 
        visualizer = DiscriminationThreshold(algorithm)
        visualizer.fit(training_x,training_y)
        visualizer.poof()

In [ ]:
x_train = pd.DataFrame(x_train, columns=X.columns)
x_test = pd.DataFrame(x_test, columns=X.columns)
x_train.head()

In [ ]:
Log_reg = LogisticRegression(C=150, max_iter=150)
Log_reg.fit(x_train, y_train)
log_pred = Log_reg.predict(x_test)

print(f'Accuracy score : {accuracy_score(log_pred, y_test)}')
print(f'Confusion matrix :\n {confusion_matrix(log_pred, y_test)}')
print(f'Classification report :\n {classification_report(log_pred, y_test)}')

In [ ]:
# Random forest classifier
Rfc = RandomForestClassifier(n_estimators=120,criterion='gini', max_depth=15, min_samples_leaf=10, min_samples_split=5)
Rfc.fit(x_train, y_train)
rfc_pred = Rfc.predict(x_test)

print(f'Accuracy score : {accuracy_score(rfc_pred, y_test)}')
print(f'Confusion matrix :\n {confusion_matrix(rfc_pred, y_test)}')
print(f'Classification report :\n {classification_report(rfc_pred, y_test)}')

In [ ]:
# decisionTree Classifier
Dtc = DecisionTreeClassifier(criterion='gini', splitter='random', min_samples_leaf=15)
Dtc.fit(x_train, y_train)
dtc_pred = Dtc.predict(x_test)

print(f'Accuracy score : {accuracy_score(dtc_pred, y_test)}')
print(f'Confusion matrix :\n {confusion_matrix(dtc_pred, y_test)}')
print(f'Classification report :\n {classification_report(dtc_pred, y_test)}')

___

**Comparé à l'ensemble de données déséquilibré, notre modèle fonctionne correctement mais n'est pas optimal pour un projet de bout en bout. Nous devons donc effectuer un suréchantillonnage des données pour réduire les vrais négatifs (TN) et les faux négatifs (FN), tout en augmentant les faux positifs (FP) et les vrais positifs (TP) afin de construire un meilleur modèle.**

____

### **Utilisation de SMOTEENN pour un ensemble de données déséquilibré :**
     Suréchantillonnage avec SMOTE et nettoyage avec ENN. Combinaison de sur-échantillonnage et de sous-échantillonnage avec SMOTE et les voisins les  plus proches édités.


In [ ]:
from imblearn.combine import SMOTEENN

# Ensure we are using the fully numeric df_processed
X = df_processed.drop(columns=['Churn'])
y = df_processed['Churn']

print("X dtypes:\n", X.dtypes)
print("Unique values in y:", y.unique())

smote_enn = SMOTEENN()
X_resampled1, y_resampled1 = smote_enn.fit_resample(X, y)
print("Resampling complete.")


In [ ]:
x_train,x_test,y_train,y_test=train_test_split(X_resampled1, y_resampled1,test_size=0.2)

In [ ]:
cols = [i for i in df.columns if i not in ['Churn']]

In [ ]:
x_train.head()

### ``logistic regression``

In [ ]:
# logistic regression
Log_reg= LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                           intercept_scaling=1, max_iter=100, multi_class='ovr', n_jobs=1,
                           penalty='l2', random_state=None, solver='liblinear', tol=0.0001,
                           verbose=0, warm_start=False)
telecom_churn_prediction(Log_reg, x_train, x_test, y_train, y_test, "coefficients", threshold_plot=True)

### ``RandomForestClassifier``

In [ ]:
random_forest = RandomForestClassifier(n_estimators = 100, random_state = 123,
                             max_depth = 9, criterion = "gini")

telecom_churn_prediction(random_forest, x_train, x_test, y_train, y_test, 'features', threshold_plot=True)
#telecom_churn_prediction(Rfc, xr_train1, xr_test1, yr_train1, yr_test1, "coefficients", threshold_plot=True)

### ``DecisionTreeClassifier``

In [ ]:
decision_tree = DecisionTreeClassifier(max_depth = 9, random_state = 123,
                                       splitter = "best", criterion = "gini")

telecom_churn_prediction(decision_tree, x_train, x_test, y_train, y_test, "features", threshold_plot=True)

In [ ]:
# GradientBoostingClassifier
gbc_tunning = GradientBoostingClassifier(criterion='squared_error', learning_rate=0.3,
                           max_depth=19, max_leaf_nodes=24, min_samples_leaf=9,
                           min_samples_split=7, n_estimators=150)

telecom_churn_prediction(gbc_tunning, x_train, x_test, y_train, y_test, "features", threshold_plot=True)

In [ ]:
from sklearn.svm import SVC
#Support vector classifier using linear hyper plane
svc_lin  = SVC(C=1.0, kernel='linear', probability=True, random_state=14)
telecom_churn_prediction(svc_lin, x_train, x_test, y_train, y_test, "coefficients", threshold_plot=True)

In [ ]:
#support vector classifier using non-linear hyper plane ("rbf")
svc_rbf  = SVC(C=10.0, kernel='rbf', gamma=0.1, probability=True, random_state=124)   

telecom_churn_prediction(svc_rbf, x_train, x_test, y_train, y_test, "None", threshold_plot=True)


In [ ]:
import pickle

# Enregistrer le modèle dans un fichier
with open('../models/random_forest_model.pkl', 'wb') as f:
    pickle.dump(random_forest, f)

In [ ]:
load_model = pickle.load(open('../models/random_forest_model.pkl', 'rb'))

In [ ]:
model_score_r1 = load_model.predict(x_test)

In [ ]:
model_score_r1

In [ ]:
x_test.head()

In [ ]:
x_test.to_csv('../data/x_test.csv', index=False)

In [ ]:
import pandas as pd
import plotly.figure_factory as ff

cols = [i for i in df.columns if i not in ['Churn']]

# Putting all the model names, model classes and the used columns in a dictionary
models = {'Logistic (Baseline)': [Log_reg, cols],
          'Decision Tree': [decision_tree, cols],
          'Random Forest': [random_forest, cols],
          'SVM (linear)': [svc_lin, cols],
          'GradientBoostingClassifier': [gbc_tunning, cols],
          }

model_performances_train = pd.DataFrame()
for name in models:
    # Use a different variable name than 'df' to avoid overwriting the main dataset
    report_df = model_report(models[name][0], x_train[models[name][1]], x_test[models[name][1]],
                      y_train, y_test, name)
    model_performances_train = pd.concat([model_performances_train, report_df], ignore_index=True)

table_train = ff.create_table(np.round(model_performances_train, 4))
py.iplot(table_train)


In [ ]:
from math import ceil
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from sklearn.metrics import confusion_matrix

def confmatplot(modeldict, df_train, df_test, target_train, target_test, figcolnumber):
    fig = plt.figure(figsize=(4*figcolnumber, 4*ceil(len(modeldict)/figcolnumber)))
    fig.set_facecolor("#F3F3F3")
    for name, figpos in itertools.zip_longest(modeldict, range(len(modeldict))):
        plt.subplot(ceil(len(modeldict)/figcolnumber), figcolnumber, figpos+1)
        # Access the first element of df_train and target_train lists
        train_df = df_train[0]
        train_target = target_train[0]
        
        model = modeldict[name][0].fit(train_df[modeldict[name][1]], train_target)
        predictions = model.predict(df_test[modeldict[name][1]])
        conf_matrix = confusion_matrix(target_test, predictions)
        sns.heatmap(conf_matrix, annot=True, fmt = "d", square = True,
                    xticklabels=["Not churn", "Churn"],
                    yticklabels=["Not churn", "Churn"],
                    linewidths = 2, linecolor = "w", cmap = "Set1")
        plt.title(name, color = "b")
        plt.subplots_adjust(wspace = .3, hspace = .3)


In [ ]:
confmatplot(modeldict=models, df_train=[x_train], df_test=x_test, 
             target_train=[y_train], target_test=y_test, figcolnumber=3)

In [ ]:
#evaluation of results
def model_evaluation(y_test, y_pred, model_name):
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    f2 = fbeta_score(y_test, y_pred, beta = 2.0)

    results = pd.DataFrame([[model_name, acc, prec, rec, f1, f2]], 
                       columns = ["Model", "Accuracy", "Precision", "Recall",
                                 "F1 SCore", "F2 Score"])
    results = results.sort_values(["Precision", "Recall", "F2 Score"], ascending = False)
    return results

In [ ]:
# Generate predictions for each model and evaluate
# We use the models trained after SMOTE resampling
lr = model_evaluation(y_test, Log_reg.predict(x_test), "Logistic Regression")
dt = model_evaluation(y_test, decision_tree.predict(x_test), "Decision Tree")
rf = model_evaluation(y_test, random_forest.predict(x_test), "Random Forest")
gb = model_evaluation(y_test, gbc_tunning.predict(x_test), "Gradient Boost")
sl = model_evaluation(y_test, svc_lin.predict(x_test), "SVM (Linear)")
sr = model_evaluation(y_test, svc_rbf.predict(x_test), "SVM (RBF)")


In [ ]:
eval_ = pd.concat([lr, dt, rf, gb, sl, sr], ignore_index=True).sort_values(["Precision", 
"Recall", "F2 Score"], ascending = False).reset_index(drop=True)
eval_


In [ ]:

summary = (df[[i for i in df.columns]].
           describe().transpose().reset_index())

summary = summary.rename(columns = {"index" : "feature"})
summary = np.around(summary,3)

val_lst = [summary['feature'], summary['count'],
           summary['mean'],summary['std'],
           summary['min'], summary['25%'],
           summary['50%'], summary['75%'], summary['max']]

trace  = go.Table(header = dict(values = summary.columns.tolist(),
                                line = dict(color = ['#506784']),
                                fill = dict(color = ['#119DFF']),
                               ),
                  cells  = dict(values = val_lst,
                                line = dict(color = ['#506784']),
                                fill = dict(color = ["lightgrey",'#F5F8FF'])
                               ),
                  columnwidth = [200,60,100,100,60,60,80,80,80])
layout = go.Layout(dict(title = "Training variable Summary"))
figure = go.Figure(data=[trace],layout=layout)
py.iplot(figure)

``` python
import pandas as pd
import pickle

# Charger le DataFrame à partir du fichier CSV
dft = pd.read_csv('Mod_tel_churn.csv')

# Charger les encodeurs de labels et le scaler à partir du fichier pickle
with open('label_encoders_and_scaler.pkl', 'rb') as f:
    label_encoders, scaler = pickle.load(f)

# Transformer les colonnes catégorielles avec les encodeurs de labels chargés
for col, encoder in label_encoders.items():
    dft[col] = encoder.transform(dft[col])

# Normaliser les colonnes numériques avec le scaler chargé
numeric_cols = dft.select_dtypes(include=['int64', 'float64']).columns
dft[numeric_cols] = scaler.transform(dft[numeric_cols])

# Maintenant, le DataFrame 'dft' contient les données transformées

```